# Unit 1, Lecture 1: Your first agent

Forty lines, no framework. Everything a framework does, you are about to do by hand.

**Before you start:** `python setup_check.py` from the repository root must print `You are ready.`

Four requirements, and one section below for each:

1. a goal, not a question
2. tools it is allowed to use
3. a loop that carries state
4. a way to stop

## 0. Which lane are you on?

In [ ]:
from cse476.lanes import get_client, MODEL, describe

# WHY: print this every time. Half of all "it stopped working" reports are
# actually "I am on a different lane than I thought".
print(describe())
client = get_client()

## 1 and 2. The tools

A tool is an ordinary Python function. No decorator, no base class, no registration
with a vendor. It is the function you would have written anyway.

In [ ]:
ROOMS = {
    ("Taj Palace", "2026-08-14"): 3,
    ("Taj Palace", "2026-08-15"): 0,
    ("Radisson Blu", "2026-08-14"): 11,
    ("Radisson Blu", "2026-08-15"): 7,
}
RATES = {"Taj Palace": 14500, "Radisson Blu": 6200}


def get_room_availability(hotel: str, date: str) -> str:
    n = ROOMS.get((hotel, date))
    if n is None:
        return f"No record for {hotel} on {date}."
    return f"{hotel} on {date}: {n} rooms available."


def get_nightly_rate(hotel: str) -> str:
    rate = RATES.get(hotel)
    if rate is None:
        return f"No rate on file for {hotel}."
    return f"{hotel}: Rs {rate} per night."


# Test tools before wiring them into an agent. Always. A tool that is broken
# inside a loop is very hard to tell apart from a model that is confused.
print(get_room_availability("Taj Palace", "2026-08-14"))
print(get_nightly_rate("Taj Palace"))

### The whitelist

`REGISTRY` is not a convenience. If a name is not a key in this dict, it does not
run, no matter what the model asks for. We will test that claim later in this notebook.

In [ ]:
REGISTRY = {
    "get_room_availability": get_room_availability,
    "get_nightly_rate": get_nightly_rate,
}

### The schema

This is the only thing the model ever sees. It cannot see your function body, your
variable names or your data. If the description is vague, the model picks the wrong
tool, and the bug is in your English rather than your Python.

In [ ]:
TOOL_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "get_room_availability",
            "description": (
                "Get the number of rooms available at a specific hotel on a "
                "specific date. Use this when asked whether a hotel has space."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "hotel": {"type": "string", "description": "Exact hotel name."},
                    "date": {"type": "string", "description": "Date as YYYY-MM-DD."},
                },
                "required": ["hotel", "date"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_nightly_rate",
            "description": (
                "Get the nightly room rate in rupees for a hotel. Use this when "
                "asked about price or cost. Does not tell you availability."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "hotel": {"type": "string", "description": "Exact hotel name."},
                },
                "required": ["hotel"],
            },
        },
    },
]

## 3 and 4. The loop

Read the highlighted comment on the `REGISTRY[name](**args)` line. That is the only
statement in this entire notebook where anything actually executes. The model never
called anything. It asked, and this loop obeyed.

In [ ]:
import json

SYSTEM = (
    "You are a hotel booking assistant. You have tools for room availability "
    "and nightly rates. Use them rather than guessing. When you have everything "
    "you need, answer the user directly and stop calling tools."
)


def run_agent(goal: str, max_steps: int = 6, verbose: bool = True) -> str:
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": goal},
    ]

    for step in range(1, max_steps + 1):
        # THINK
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOL_SCHEMA
        )
        message = response.choices[0].message

        # WHY: the assistant turn goes back in whether or not it asked for a tool.
        # Drop it and the model loses the thread of its own reasoning.
        messages.append(message.model_dump(exclude_none=True))

        # EXIT 1: the model stopped asking for tools.
        if not message.tool_calls:
            if verbose:
                print(f"[step {step}] done")
            return message.content or ""

        # ACT. The model did not call anything. It returned a request.
        for call in message.tool_calls:
            name = call.function.name
            args = json.loads(call.function.arguments or "{}")

            if name in REGISTRY:
                result = REGISTRY[name](**args)   # <-- the only executing line
            else:
                result = f"Error: no tool named '{name}'. Available: {list(REGISTRY)}"

            if verbose:
                print(f"[step {step}] {name}({args}) -> {result}")

            # OBSERVE
            messages.append(
                {"role": "tool", "tool_call_id": call.id, "content": result}
            )

    # EXIT 2: the budget ran out. Say so honestly.
    return f"Stopped after {max_steps} steps without reaching a final answer."


print("agent defined")

## Run it

Nowhere did we say "check availability first, then price". Watch the model decide
that order for itself. Then change the question to ask only about price, and watch
step 1 disappear.

In [ ]:
answer = run_agent(
    "Can I get a room at Taj Palace on 2026-08-14, and what would it cost?"
)
print()
print(answer)

In [ ]:
# Run it again with a different shape of question. Count the steps.
print(run_agent("How much is the Radisson Blu per night?"))

## Break it, failure one: no way to stop

Give it a goal none of its tools can satisfy, with a generous budget, and watch it
keep trying. Every attempt below is a billed API call.

**Interrupt the kernel when you have seen enough.** That physical act is the lesson.

In [ ]:
# The agent is being helpful, not broken. That is what makes this expensive.
print(run_agent("Book me a room at the Oberoi in Shimla for tonight", max_steps=8))

The fix is the `for step in range(1, max_steps + 1)` line you already have.

It does not solve the underlying problem. The agent still cannot answer. What it does
is turn an unbounded failure into a bounded one, so the worst case is arithmetic you
can do in advance rather than a number you discover on an invoice.

## Break it, failure two: an invented tool

We do not touch the code this time. We just ask for something the model would like a
tool for, and does not have. Watch it request `send_email`, which does not exist.

In [ ]:
print(run_agent("Check the Taj Palace on 2026-08-14 and email the result to the manager"))

That did not crash because of four lines:

```python
if name in REGISTRY:
    result = REGISTRY[name](**args)
else:
    result = f"Error: no tool named '{name}'."
```

Without the check, `REGISTRY[name]` raises `KeyError`, and in a web service that is a
500 to your user, caused by a language model saying a word.

**The principle, which returns in Unit 6:** anything the model produces is untrusted
input. A tool name is a string that came from a language model. Validate it as
carefully as a form field from the open internet.

## Your turn

Add a third tool. Something the hotel assistant would plausibly need, for example
`get_cancellation_policy(hotel)` or `get_distance_from_airport(hotel)`.

You must do three things, and forgetting the third is the most common mistake:

1. Write the function
2. Add it to `REGISTRY`
3. Add its schema to `TOOL_SCHEMA`

Then ask a question that needs all three tools, and check the step count.

**Then try this:** write the description deliberately vaguely, for example just
`"Get information about a hotel."` Ask a question about price. Watch which tool the
model picks. That is the tool description acting as an instruction rather than as
documentation.

In [ ]:
# your third tool here
